# Tweets Sentiment (Basic)

Notebook ini untuk klasifikasi sentimen dasar berbasis data tweet dari Excel Anda.

## Sumber data
- File: 08-Text Mining/tweets_clf.xlsx
- Sheet default: sheet pertama

## Target belajar
1. Membaca data teks dari Excel
2. Cleaning teks sederhana
3. Membuat fitur TF-IDF
4. Melatih model klasifikasi sederhana
5. Evaluasi akurasi dasar

## 1) Install dan import library

In [ ]:
# Uncomment jika library belum ada
# !pip install pandas openpyxl scikit-learn

In [ ]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

## 2) Baca data dari Excel

In [ ]:
path_file = "tweets_clf.xlsx"
df = pd.read_excel(path_file)
print("Ukuran data:", df.shape)
df.head()

## 3) Deteksi kolom teks dan label
Jika otomatis tidak cocok, silakan set manual text_col dan label_col.

In [ ]:
kandidat_text = ["tweet", "text", "content", "ulasan", "komentar"]
kandidat_label = ["label", "sentiment", "sentimen", "kelas", "target"]

lower_map = {c.lower(): c for c in df.columns}

text_col = next((lower_map[k] for k in kandidat_text if k in lower_map), None)
label_col = next((lower_map[k] for k in kandidat_label if k in lower_map), None)

print("Kolom terdeteksi -> text:", text_col, "| label:", label_col)
print("Daftar kolom:", df.columns.tolist())

## 4) Cleaning teks sederhana

In [ ]:
def clean_text(s):
    s = str(s).lower()
    s = re.sub(r"http\S+|www\S+", " ", s)
    s = re.sub(r"@\w+", " ", s)
    s = re.sub(r"#", " ", s)
    s = re.sub(r"[^a-zA-Z\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df_work = df.copy()
df_work = df_work.dropna(subset=[text_col, label_col])
df_work["clean_text"] = df_work[text_col].apply(clean_text)

df_work[[text_col, "clean_text", label_col]].head()

## 5) Split data train-test
Kita pakai 80% data untuk training, 20% untuk testing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_work["clean_text"],
    df_work[label_col],
    test_size=0.2,
    random_state=42,
    stratify=df_work[label_col]
)

print("Jumlah train:", len(X_train))
print("Jumlah test:", len(X_test))

## 6) TF-IDF + Naive Bayes
Ini baseline yang ringan dan mudah dipahami pemula.

In [ ]:
vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 1))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

## 7) Evaluasi model

In [ ]:
acc = accuracy_score(y_test, y_pred)
print("Accuracy:", round(acc, 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## 8) Uji 1 kalimat baru

In [ ]:
contoh_tweet = "pelayanan hari ini sangat cepat dan memuaskan"
contoh_bersih = clean_text(contoh_tweet)
contoh_vec = vectorizer.transform([contoh_bersih])
prediksi = model.predict(contoh_vec)[0]

print("Tweet:", contoh_tweet)
print("Prediksi sentimen:", prediksi)

## 9) Simpan hasil prediksi pada seluruh data

In [ ]:
X_all = vectorizer.transform(df_work["clean_text"])
df_work["prediksi_sentimen"] = model.predict(X_all)

output_file = "08-Text Mining/tweets_sentiment_result.csv"
df_work.to_csv(output_file, index=False)
print("Hasil tersimpan di:", output_file)